# 01 — EDA and a naive baseline

**The exploratory phase.** A data scientist gets the California Housing task:
predict median house value (in $100k) from census block features. This notebook is
deliberately messy-by-design — quick looks, a hand-rolled baseline, ad-hoc MLflow
logging. Nothing here is production code.

Runs log to the **Azure ML workspace** — browse them in Studio (ml.azure.com → Jobs).
Set the tracking URI before starting Jupyter (needs `az login`):

```bash
export MLFLOW_TRACKING_URI=$(make -s azure-uri)
```

Offline alternative: `make mlflow-up` starts a local throwaway server on :5050
(the fallback below).

In [ ]:
import os

os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5050")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

bunch = fetch_california_housing(as_frame=True)
df = bunch.frame
df.head()

In [ ]:
df.describe().T

The target `MedHouseVal` is capped at 5.0 ($500k) — visible as a spike in the
distribution. Worth knowing before trusting RMSE too much at the high end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["MedHouseVal"].hist(bins=60, ax=axes[0])
axes[0].set_title("Target: MedHouseVal ($100k)")
df["MedInc"].hist(bins=60, ax=axes[1])
axes[1].set_title("Strongest feature: MedInc")
plt.tight_layout()

In [ ]:
# Correlation of each feature with the target — MedInc dominates.
df.corr(numeric_only=True)["MedHouseVal"].drop("MedHouseVal").sort_values().plot.barh(figsize=(6, 3.5))
plt.title("Feature correlation with target")
plt.tight_layout()

## Naive baseline: scaled Ridge

Hand-rolled, in-notebook. Same split convention the team uses everywhere:
`test_size=0.2, seed=42`.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = bunch.data, bunch.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
r2 = float(r2_score(y_test, pred))
print(f"baseline ridge: rmse={rmse:.4f}  r2={r2:.4f}")

In [ ]:
# Ad-hoc logging so runs aren't lost between sessions. With the Azure tracking
# URI exported, this lands in the workspace; otherwise the local dev server.
import mlflow

mlflow.set_experiment("notebook-exploration")
with mlflow.start_run(run_name="ridge_baseline"):
    mlflow.log_params({"model": "ridge", "alpha": 1.0, "seed": 42})
    mlflow.log_metrics({"rmse": rmse, "r2": r2})
print(f"logged to {mlflow.get_tracking_uri()[:60]}...")

## Where this goes next

RMSE ≈ 0.75 is a serviceable baseline but linear models can't capture the
location/interaction structure. Next notebook: prototype XGBoost, then hand the
experiment off to the team's config-driven package so CI can own it
(`02_prototype_to_package.ipynb`).